# 03b — Audit del target: Tumor + Healthy

## Obiettivo

Verificare se il task di classificazione utilizzato finora coincide con
l'obiettivo del progetto: associare un sample eccDNA al tipo di tumore,
includendo anche la classe Healthy.

Il dataset completo contiene 18 classi, alcune delle quali sembrano
rappresentare patologie non oncologiche.

Prima di modificare il dataset viene quindi analizzata la variabile
`disease_group` per verificare come le classi siano categorizzate
nel metadata originale.

Il dataset originale non viene modificato.
Eventualmente verrà creato un nuovo manifest contenente solamente:

- classi appartenenti al gruppo cancer;
- classe Healthy.

Lo split basato su `split_cluster` viene mantenuto invariato.

In [1]:
# ============================================================
# CELL 2 — IMPORT E DATA
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)


metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


print(
    "Sample totali:",
    len(metadata)
)

print(
    "Classi:",
    metadata["class_id"].nunique()
)

print(
    "Disease group:",
    metadata["disease_group"].unique()
)

Sample totali: 665681
Classi: 18
Disease group: <StringArray>
['cancer', 'healthy', 'non_cancer_disease']
Length: 3, dtype: str


In [2]:
# ============================================================
# CELL 3 — AUDIT DISEASE GROUP
# ============================================================

disease_audit_df = (

    metadata[
        [
            "class_id",
            "disease",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        [
            "class_id",
            "disease_clean"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    disease_audit_df
)

,class_id,disease,disease_clean,disease_group
0,0,Gastric cancer,gastric cancer,cancer
1,0,Stomach,gastric cancer,cancer
2,0,Stomach Cancer,gastric cancer,cancer
3,1,Health,healthy,healthy
4,1,Healthy,healthy,healthy
5,2,Ovarian Cancer,ovarian cancer,cancer
6,2,Ovarian cancer,ovarian cancer,cancer
7,3,Prostate Cancer,prostate cancer,cancer
8,3,Prostate cancer,prostate cancer,cancer
9,4,Colorectal cancer,colorectal cancer,cancer


In [3]:
# ============================================================
# CELL 4 — CONSISTENZA CLASS_ID / DISEASE_GROUP
# ============================================================

group_consistency = (

    metadata
    .groupby("class_id")
    .agg(
        disease_clean=(
            "disease_clean",
            lambda x: sorted(
                x.dropna().unique()
            )
        ),
        disease_group=(
            "disease_group",
            lambda x: sorted(
                x.dropna().unique()
            )
        ),
        n_samples=(
            "id",
            "size"
        )
    )
    .reset_index()
)


display(
    group_consistency
)


problematic_groups = (
    group_consistency[
        group_consistency[
            "disease_group"
        ].apply(len) != 1
    ]
)


print(
    "Classi con disease_group ambiguo:",
    len(problematic_groups)
)

,class_id,disease_clean,disease_group,n_samples
0,0,[gastric cancer],[cancer],391239
1,1,[healthy],[healthy],73995
2,2,[ovarian cancer],[cancer],35122
3,3,[prostate cancer],[cancer],34487
4,4,[colorectal cancer],[cancer],22933
5,5,[lymphoma],[cancer],20185
6,6,[hiv infectious disease],[non_cancer_disease],16315
7,7,[cervical adenocarcinoma],[cancer],15677
8,8,[leukemia],[cancer],13456
9,9,[primary pulmonary hypertension],[non_cancer_disease],8119


Classi con disease_group ambiguo: 0


In [4]:
# ============================================================
# CELL 5 — DISTRIBUZIONE DEI GRUPPI
# ============================================================

group_summary = (

    metadata
    .groupby(
        "disease_group"
    )
    .agg(
        n_samples=(
            "id",
            "size"
        ),
        n_classes=(
            "class_id",
            "nunique"
        )
    )
    .sort_values(
        "n_samples",
        ascending=False
    )
    .reset_index()
)


display(
    group_summary
)

,disease_group,n_samples,n_classes
0,cancer,551942,11
1,healthy,73995,1
2,non_cancer_disease,39744,6


In [5]:
# ============================================================
# CELL 6 — TUMOR + HEALTHY TASK
# ============================================================

TARGET_GROUPS = [
    "cancer",
    "healthy"
]


tumor_healthy_mask = (
    metadata["disease_group"]
    .isin(TARGET_GROUPS)
)


tumor_healthy_metadata = (
    metadata[
        tumor_healthy_mask
    ]
    .copy()
    .reset_index(drop=True)
)


excluded_metadata = (
    metadata[
        ~tumor_healthy_mask
    ]
    .copy()
)


print("=" * 70)
print("TUMOR + HEALTHY TASK")
print("=" * 70)

print(
    "Sample inclusi:",
    len(tumor_healthy_metadata)
)

print(
    "Classi incluse:",
    tumor_healthy_metadata[
        "class_id"
    ].nunique()
)

print()

print(
    "Sample esclusi:",
    len(excluded_metadata)
)

print(
    "Classi escluse:",
    excluded_metadata[
        "class_id"
    ].nunique()
)

TUMOR + HEALTHY TASK
Sample inclusi: 625937
Classi incluse: 12

Sample esclusi: 39744
Classi escluse: 6


In [6]:
# ============================================================
# CELL 7 — INCLUDED / EXCLUDED CLASSES
# ============================================================

included_classes_df = (

    tumor_healthy_metadata[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "class_id"
    )

    .reset_index(
        drop=True
    )
)


excluded_classes_df = (

    excluded_metadata[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "class_id"
    )

    .reset_index(
        drop=True
    )
)


print("CLASSI INCLUSE")
display(
    included_classes_df
)


print()
print("CLASSI ESCLUSE")
display(
    excluded_classes_df
)

CLASSI INCLUSE


,class_id,disease_clean,disease_group
0,0,gastric cancer,cancer
1,1,healthy,healthy
2,2,ovarian cancer,cancer
3,3,prostate cancer,cancer
4,4,colorectal cancer,cancer
5,5,lymphoma,cancer
6,7,cervical adenocarcinoma,cancer
7,8,leukemia,cancer
8,11,hypopharyngeal squamous cell carcinoma,cancer
9,12,glioblastoma cancer,cancer



CLASSI ESCLUSE


,class_id,disease_clean,disease_group
0,6,hiv infectious disease,non_cancer_disease
1,9,primary pulmonary hypertension,non_cancer_disease
2,10,cataract,non_cancer_disease
3,14,dilated cardiomyopathy,non_cancer_disease
4,16,chronic kidney disease,non_cancer_disease
5,17,branchio-oculo-facial syndrome (bofs),non_cancer_disease


In [7]:
# ============================================================
# CELL 8 — NEW CLASS MAPPING
# ============================================================

class_mapping = (

    included_classes_df[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .rename(
        columns={
            "class_id":
                "original_class_id"
        }
    )

    .sort_values(
        "original_class_id"
    )

    .reset_index(
        drop=True
    )
)


class_mapping[
    "class_id"
] = np.arange(
    len(class_mapping),
    dtype=np.int64
)


old_to_new_class_id = dict(
    zip(
        class_mapping[
            "original_class_id"
        ],
        class_mapping[
            "class_id"
        ]
    )
)


display(
    class_mapping[
        [
            "class_id",
            "original_class_id",
            "disease_clean",
            "disease_group"
        ]
    ]
)

,class_id,original_class_id,disease_clean,disease_group
0,0,0,gastric cancer,cancer
1,1,1,healthy,healthy
2,2,2,ovarian cancer,cancer
3,3,3,prostate cancer,cancer
4,4,4,colorectal cancer,cancer
5,5,5,lymphoma,cancer
6,6,7,cervical adenocarcinoma,cancer
7,7,8,leukemia,cancer
8,8,11,hypopharyngeal squamous cell carcinoma,cancer
9,9,12,glioblastoma cancer,cancer


In [8]:
# ============================================================
# CELL 9 — BUILD TUMOR + HEALTHY MANIFEST
# ============================================================

tumor_healthy_manifest = (
    tumor_healthy_metadata
    .copy()
)


# Manteniamo traccia della label originale
tumor_healthy_manifest[
    "original_class_id"
] = (
    tumor_healthy_manifest[
        "class_id"
    ]
)


# Nuova label 0...11
tumor_healthy_manifest[
    "class_id"
] = (

    tumor_healthy_manifest[
        "original_class_id"
    ]

    .map(
        old_to_new_class_id
    )

    .astype(
        np.int64
    )
)


print(
    "Classi finali:",
    tumor_healthy_manifest[
        "class_id"
    ].nunique()
)


display(
    tumor_healthy_manifest[
        [
            "id",
            "disease_clean",
            "disease_group",
            "original_class_id",
            "class_id",
            "split_cluster"
        ]
    ]
    .head()
)

Classi finali: 12


,id,disease_clean,disease_group,original_class_id,class_id,split_cluster
0,CircleBaseV2_000726522,gastric cancer,cancer,0,0,train
1,CircleBaseV2_000119689,gastric cancer,cancer,0,0,train
2,CircleBaseV2_002112184,gastric cancer,cancer,0,0,train
3,CircleBaseV2_001452203,gastric cancer,cancer,0,0,train
4,CircleBaseV2_000827693,gastric cancer,cancer,0,0,train


In [9]:
# ============================================================
# CELL 10 — SPLIT AUDIT
# ============================================================

split_summary = (

    tumor_healthy_manifest

    .groupby(
        [
            "class_id",
            "disease_clean",
            "split_cluster"
        ]
    )

    .size()

    .rename(
        "n_samples"
    )

    .reset_index()
)


split_pivot = (

    split_summary

    .pivot_table(
        index=[
            "class_id",
            "disease_clean"
        ],
        columns=
            "split_cluster",
        values=
            "n_samples",
        fill_value=0
    )

    .reset_index()
)


display(
    split_pivot
)

split_cluster,class_id,disease_clean,test,train,val
0,0,gastric cancer,128668.0,10000.0,252571.0
1,1,healthy,20852.0,10000.0,43143.0
2,2,ovarian cancer,8355.0,10000.0,16767.0
3,3,prostate cancer,8263.0,10000.0,16224.0
4,4,colorectal cancer,4422.0,10000.0,8511.0
5,5,lymphoma,3409.0,10000.0,6776.0
6,6,cervical adenocarcinoma,1958.0,10000.0,3719.0
7,7,leukemia,1138.0,10000.0,2318.0
8,8,hypopharyngeal squamous cell carcinoma,288.0,5052.0,585.0
9,9,glioblastoma cancer,296.0,4880.0,505.0


In [10]:
# ============================================================
# CELL 11 — SAVE TUMOR + HEALTHY MANIFEST
# ============================================================

TUMOR_HEALTHY_MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


TUMOR_HEALTHY_CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


tumor_healthy_manifest.to_csv(
    TUMOR_HEALTHY_MANIFEST_PATH,
    sep="\t",
    index=False
)


class_mapping.to_csv(
    TUMOR_HEALTHY_CLASS_MAPPING_PATH,
    sep="\t",
    index=False
)


print(
    "Manifest:",
    TUMOR_HEALTHY_MANIFEST_PATH
)

print(
    "Mapping:",
    TUMOR_HEALTHY_CLASS_MAPPING_PATH
)

Manifest: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_tumor_healthy_manifest.tsv
Mapping: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_tumor_healthy_class_mapping.tsv


In [11]:
# ============================================================
# CELL 12 — LOAD VALIDATION POOL + EUCLIDEAN V3
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


FCGR_MEMMAP_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k6.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k6_index.tsv"
)


EUCLIDEAN_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_euclidean_v3"
    / "euclidean_v3_margin_1p25_best.pt"
)


val_pool_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_pool_original["class_id"] = (
    val_pool_original["class_id"]
    .astype(int)
)


print(
    "Validation pool originale:",
    len(val_pool_original)
)

print(
    "Classi originali:",
    val_pool_original[
        "class_id"
    ].nunique()
)

print(
    "Checkpoint trovato:",
    EUCLIDEAN_CHECKPOINT_PATH.exists()
)

Validation pool originale: 12937
Classi originali: 18
Checkpoint trovato: True


In [12]:
# ============================================================
# CELL 13 — FILTER VALIDATION TO TUMOR + HEALTHY
# ============================================================

included_original_class_ids = (
    class_mapping[
        "original_class_id"
    ]
    .astype(int)
    .tolist()
)


tumor_val_pool = (
    val_pool_original[
        val_pool_original[
            "class_id"
        ].isin(
            included_original_class_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# Salviamo l'ID originale
tumor_val_pool[
    "original_class_id"
] = (
    tumor_val_pool[
        "class_id"
    ]
)


# Rimappiamo 0...11
tumor_val_pool[
    "class_id"
] = (
    tumor_val_pool[
        "original_class_id"
    ]
    .map(
        old_to_new_class_id
    )
    .astype(int)
)


print(
    "Validation Tumor + Healthy:",
    len(tumor_val_pool)
)

print(
    "Numero classi:",
    tumor_val_pool[
        "class_id"
    ].nunique()
)


display(
    tumor_val_pool[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_val")
    .to_frame()
)

Validation Tumor + Healthy: 9753
Numero classi: 12


,n_val
class_id,
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,585


In [13]:
# ============================================================
# CELL 14 — BALANCED TRAIN REFERENCE SET
# ============================================================

tumor_train = (
    tumor_healthy_manifest[
        tumor_healthy_manifest[
            "split_cluster"
        ] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Train Tumor + Healthy:",
    len(tumor_train)
)

print(
    "Classi:",
    tumor_train[
        "class_id"
    ].nunique()
)


display(
    tumor_train[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_train")
    .to_frame()
)

Train Tumor + Healthy: 96167
Classi: 12


,n_train
class_id,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,5052


In [14]:
# ============================================================
# CELL 15 — FCGR DATASET
# ============================================================

fcgr_memmap = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


class SingleSampleDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata[
                "class_id"
            ]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )


        x = np.array(
            self.fcgr_memmap[row],
            dtype=np.float32,
            copy=True
        )


        return {
            "x":
                torch.from_numpy(
                    x
                ).unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }

In [15]:
# ============================================================
# CELL 16 — EUCLIDEAN V3 ENCODER
# ============================================================

class FCGRCNNEncoderV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1, 32, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32, 32, 3,
                padding=1,
                bias=False
            ),

            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(x)

        z = self.embedding_head(x)

        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )


class SiameseNetworkV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderV3(
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        return self.encoder(x)

In [16]:
# ============================================================
# CELL 17 — LOAD PRETRAINED EUCLIDEAN V3
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


checkpoint = torch.load(
    EUCLIDEAN_CHECKPOINT_PATH,
    map_location=DEVICE
)


model = (
    SiameseNetworkV3(
        embedding_dim=128
    )
    .to(DEVICE)
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)


model.eval()


print(
    "Best epoch:",
    checkpoint[
        "best_epoch"
    ]
)

print(
    "Best pairwise Val AUC:",
    checkpoint[
        "best_val_auc"
    ]
)

print(
    "Modello caricato: OK"
)

Best epoch: 38
Best pairwise Val AUC: 0.67727181092637
Modello caricato: OK


In [17]:
# ============================================================
# CELL 18 — LOADERS
# ============================================================

BATCH_SIZE = 256


train_dataset = SingleSampleDataset(
    tumor_train,
    fcgr_memmap,
    id_to_fcgr_row
)


val_dataset = SingleSampleDataset(
    tumor_val_pool,
    fcgr_memmap,
    id_to_fcgr_row
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

Train samples: 96167
Validation samples: 9753


In [18]:
# ============================================================
# CELL 19 — EXTRACT EMBEDDINGS
# ============================================================

def extract_embeddings(
    model,
    loader
):

    embeddings_list = []
    labels_list = []


    model.eval()


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                z = model(x)


            embeddings_list.append(
                z.float()
                .cpu()
                .numpy()
            )


            labels_list.append(
                batch[
                    "class_id"
                ]
                .numpy()
            )


    return (
        np.concatenate(
            embeddings_list,
            axis=0
        ),

        np.concatenate(
            labels_list,
            axis=0
        )
    )


print(
    "Estrazione train..."
)

train_embeddings, train_labels = (
    extract_embeddings(
        model,
        train_loader
    )
)


print(
    "Estrazione validation..."
)

val_embeddings, val_labels = (
    extract_embeddings(
        model,
        val_loader
    )
)


print(
    "Train embedding:",
    train_embeddings.shape
)

print(
    "Val embedding:",
    val_embeddings.shape
)

Estrazione train...
Estrazione validation...
Train embedding: (96167, 128)
Val embedding: (9753, 128)


In [19]:
# ============================================================
# CELL 20 — 12-WAY TUMOR + HEALTHY CLASSIFICATION
# ============================================================

classes = np.array(
    sorted(
        np.unique(
            train_labels
        )
    ),
    dtype=np.int64
)


prototypes = []


for class_id in classes:

    class_embeddings = (
        train_embeddings[
            train_labels == class_id
        ]
    )


    prototype = (
        class_embeddings
        .mean(axis=0)
    )


    prototype = (
        prototype
        /
        (
            np.linalg.norm(
                prototype
            )
            +
            1e-12
        )
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes,
    axis=0
)


distances = np.sqrt(
    (
        (
            val_embeddings[:, None, :]
            -
            prototypes[None, :, :]
        )
        ** 2
    )
    .sum(axis=2)
)


prediction_index = (
    distances.argmin(
        axis=1
    )
)


y_pred = (
    classes[
        prediction_index
    ]
)


tumor_12way_metrics = {

    "accuracy":
        accuracy_score(
            val_labels,
            y_pred
        ),

    "macro_f1":
        f1_score(
            val_labels,
            y_pred,
            average="macro",
            zero_division=0
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            val_labels,
            y_pred
        )
}


print("=" * 70)
print("12-WAY TUMOR + HEALTHY")
print("=" * 70)

print(
    "Accuracy:",
    f"{tumor_12way_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{tumor_12way_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{tumor_12way_metrics['balanced_accuracy']:.4f}"
)

12-WAY TUMOR + HEALTHY
Accuracy: 0.2808
Macro-F1: 0.2481
Balanced Accuracy: 0.3025


In [20]:
# ============================================================
# CELL 21 — BALANCED EMBEDDING DATASET FOR LINEAR PROBE
# ============================================================

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


PROBE_SAMPLES_PER_CLASS = 2000

PROBE_SEED = 12345


rng_probe = np.random.default_rng(
    PROBE_SEED
)


probe_indices = []


for class_id in classes:

    class_indices = np.where(
        train_labels == class_id
    )[0]


    print(
        f"Classe {class_id}:",
        len(class_indices),
        "embedding disponibili"
    )


    if (
        len(class_indices)
        <
        PROBE_SAMPLES_PER_CLASS
    ):

        raise ValueError(
            f"Classe {class_id}: "
            f"solo {len(class_indices)} sample."
        )


    selected = rng_probe.choice(

        class_indices,

        size=
            PROBE_SAMPLES_PER_CLASS,

        replace=False
    )


    probe_indices.extend(
        selected.tolist()
    )


probe_indices = np.asarray(
    probe_indices,
    dtype=np.int64
)


rng_probe.shuffle(
    probe_indices
)


X_probe_train = (
    train_embeddings[
        probe_indices
    ]
    .astype(np.float32)
)


y_probe_train = (
    train_labels[
        probe_indices
    ]
    .astype(np.int64)
)


X_probe_val = (
    val_embeddings
    .astype(np.float32)
)


y_probe_val = (
    val_labels
    .astype(np.int64)
)


print()
print(
    "Probe train:",
    X_probe_train.shape
)

print(
    "Probe validation:",
    X_probe_val.shape
)

print(
    "Numero classi:",
    len(
        np.unique(
            y_probe_train
        )
    )
)

Classe 0: 10000 embedding disponibili
Classe 1: 10000 embedding disponibili
Classe 2: 10000 embedding disponibili
Classe 3: 10000 embedding disponibili
Classe 4: 10000 embedding disponibili
Classe 5: 10000 embedding disponibili
Classe 6: 10000 embedding disponibili
Classe 7: 10000 embedding disponibili
Classe 8: 5052 embedding disponibili
Classe 9: 4880 embedding disponibili
Classe 10: 4124 embedding disponibili
Classe 11: 2111 embedding disponibili

Probe train: (24000, 128)
Probe validation: (9753, 128)
Numero classi: 12


In [21]:
# ============================================================
# CELL 22 — LINEAR PROBE DATALOADERS
# ============================================================

PROBE_BATCH_SIZE = 512


probe_train_dataset = TensorDataset(

    torch.from_numpy(
        X_probe_train
    ),

    torch.from_numpy(
        y_probe_train
    )
)


probe_val_dataset = TensorDataset(

    torch.from_numpy(
        X_probe_val
    ),

    torch.from_numpy(
        y_probe_val
    )
)


probe_train_loader = DataLoader(

    probe_train_dataset,

    batch_size=
        PROBE_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


probe_val_loader = DataLoader(

    probe_val_dataset,

    batch_size=
        PROBE_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train batches:",
    len(probe_train_loader)
)

print(
    "Validation batches:",
    len(probe_val_loader)
)

Train batches: 47
Validation batches: 20


In [22]:
# ============================================================
# CELL 23 — LINEAR PROBE
# ============================================================

N_CLASSES = len(
    classes
)


class LinearProbe(nn.Module):

    def __init__(
        self,
        embedding_dim,
        n_classes
    ):

        super().__init__()

        self.classifier = nn.Linear(
            embedding_dim,
            n_classes
        )


    def forward(
        self,
        x
    ):

        return self.classifier(
            x
        )


linear_probe = LinearProbe(

    embedding_dim=
        train_embeddings.shape[1],

    n_classes=
        N_CLASSES

).to(
    DEVICE
)


probe_optimizer = torch.optim.AdamW(

    linear_probe.parameters(),

    lr=1e-3,

    weight_decay=1e-4
)


probe_criterion = nn.CrossEntropyLoss()


print(
    linear_probe
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in linear_probe.parameters()
    )
)

LinearProbe(
  (classifier): Linear(in_features=128, out_features=12, bias=True)
)
Trainable parameters: 1548


In [23]:
# ============================================================
# CELL 24 — LINEAR PROBE TRAIN / EVAL
# ============================================================

def train_probe_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0.0
    total_samples = 0


    for embeddings, labels in loader:

        embeddings = embeddings.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            embeddings
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()

        optimizer.step()


        batch_size = (
            labels.shape[0]
        )


        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_samples += (
            batch_size
        )


    return (
        total_loss
        /
        total_samples
    )


def evaluate_probe(
    model,
    loader
):

    model.eval()


    all_true = []
    all_pred = []


    with torch.no_grad():

        for embeddings, labels in loader:

            embeddings = embeddings.to(
                DEVICE,
                non_blocking=True
            )


            logits = model(
                embeddings
            )


            predictions = (
                logits.argmax(
                    dim=1
                )
                .cpu()
                .numpy()
            )


            all_pred.append(
                predictions
            )


            all_true.append(
                labels.numpy()
            )


    y_true = np.concatenate(
        all_true
    )


    y_pred = np.concatenate(
        all_pred
    )


    metrics = {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            )
    }


    return (
        metrics,
        y_true,
        y_pred
    )

In [24]:
# ============================================================
# CELL 25 — LINEAR PROBE TRAINING
# ============================================================

PROBE_MAX_EPOCHS = 50

PROBE_PATIENCE = 7

PROBE_MIN_DELTA = 1e-4


best_probe_f1 = -np.inf

best_probe_epoch = 0

best_probe_state = None

epochs_without_improvement = 0


probe_history = []


print("=" * 74)
print("FROZEN EMBEDDING LINEAR PROBE")
print("=" * 74)


for epoch in range(
    1,
    PROBE_MAX_EPOCHS + 1
):

    train_loss = train_probe_epoch(

        model=
            linear_probe,

        loader=
            probe_train_loader,

        optimizer=
            probe_optimizer,

        criterion=
            probe_criterion
    )


    (
        val_metrics,
        _,
        _
    ) = evaluate_probe(

        model=
            linear_probe,

        loader=
            probe_val_loader
    )


    current_f1 = float(
        val_metrics[
            "macro_f1"
        ]
    )


    improved = (
        current_f1
        >
        best_probe_f1
        +
        PROBE_MIN_DELTA
    )


    if improved:

        best_probe_f1 = (
            current_f1
        )

        best_probe_epoch = (
            epoch
        )

        best_probe_state = {
            key:
                value.detach().cpu().clone()

            for key, value
            in linear_probe.state_dict().items()
        }


        epochs_without_improvement = 0


    else:

        epochs_without_improvement += 1


    probe_history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "accuracy":
                val_metrics[
                    "accuracy"
                ],

            "macro_f1":
                val_metrics[
                    "macro_f1"
                ],

            "balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ]
        }
    )


    marker = (
        " *BEST*"
        if improved
        else ""
    )


    print(

        f"Epoch {epoch:02d}"

        f" | loss "
        f"{train_loss:.4f}"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f"{marker}"
    )


    if (
        epochs_without_improvement
        >=
        PROBE_PATIENCE
    ):

        print()
        print(
            "Early stopping."
        )

        break


# ============================================================
# RESTORE BEST
# ============================================================

linear_probe.load_state_dict(
    best_probe_state
)


(
    best_probe_metrics,
    probe_y_true,
    probe_y_pred
) = evaluate_probe(

    model=
        linear_probe,

    loader=
        probe_val_loader
)


print()
print("=" * 74)
print("BEST LINEAR PROBE")
print("=" * 74)

print(
    "Best epoch:",
    best_probe_epoch
)

print(
    "Accuracy:",
    f"{best_probe_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{best_probe_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{best_probe_metrics['balanced_accuracy']:.4f}"
)

FROZEN EMBEDDING LINEAR PROBE
Epoch 01 | loss 2.4411 | val Acc 0.2653 | val F1 0.2126 | bal Acc 0.2424 *BEST*
Epoch 02 | loss 2.3601 | val Acc 0.2914 | val F1 0.2602 | bal Acc 0.2990 *BEST*
Epoch 03 | loss 2.2912 | val Acc 0.2958 | val F1 0.2537 | bal Acc 0.2991
Epoch 04 | loss 2.2329 | val Acc 0.2971 | val F1 0.2570 | bal Acc 0.3024
Epoch 05 | loss 2.1836 | val Acc 0.2873 | val F1 0.2532 | bal Acc 0.3044
Epoch 06 | loss 2.1418 | val Acc 0.2916 | val F1 0.2612 | bal Acc 0.3090 *BEST*
Epoch 07 | loss 2.1063 | val Acc 0.2852 | val F1 0.2488 | bal Acc 0.3063
Epoch 08 | loss 2.0762 | val Acc 0.2912 | val F1 0.2608 | bal Acc 0.3090
Epoch 09 | loss 2.0507 | val Acc 0.2917 | val F1 0.2590 | bal Acc 0.3089
Epoch 10 | loss 2.0288 | val Acc 0.2900 | val F1 0.2589 | bal Acc 0.3090
Epoch 11 | loss 2.0101 | val Acc 0.2932 | val F1 0.2602 | bal Acc 0.3100
Epoch 12 | loss 1.9942 | val Acc 0.2931 | val F1 0.2577 | bal Acc 0.3106
Epoch 13 | loss 1.9804 | val Acc 0.2951 | val F1 0.2632 | bal Acc 0.3109 